In [0]:
fact_table = "adb_adventureworks_project.gold.fact_sales"

df = spark.table(fact_table)

total_rows = df.count()

unique_lines = (
    df.select("OrderNumber", "OrderLineItem")
      .distinct()
      .count()
)

assert total_rows == unique_lines, (
    f"Duplicate sales lines detected! "
    f"Total: {total_rows}, Unique: {unique_lines}"
)

print(f"PASS: {total_rows} unique sales lines")


In [0]:

from pyspark.sql import functions as F

# Calculate expected revenue from Silver
silver_sales = spark.table(
    "adb_adventureworks_project.silver.sales_silver_table"
)

silver_products = spark.table(
    "adb_adventureworks_project.silver.products_silver_table"
)

expected_revenue = (
    silver_sales.alias("s")
    .join(
        silver_products.alias("p"),
        F.col("s.ProductKey") == F.col("p.ProductKey"),
        "inner"
    )
    .select(
        (F.col("s.OrderQuantity") * F.col("p.ProductPrice")).cast("decimal(18,4)").alias("Revenue")
    )
    .agg(F.sum("Revenue").alias("TotalRevenue"))
    .first()["TotalRevenue"]
)

# Calculate actual revenue from Gold
actual_revenue = (
    spark.table("adb_adventureworks_project.gold.fact_sales")
    .agg(F.sum("EstimatedRevenue").alias("TotalRevenue"))
    .first()["TotalRevenue"]
)

# Compare at the cent level
assert round(expected_revenue, 2) == round(actual_revenue, 2), (
    f"Revenue mismatch! "
    f"Silver: {expected_revenue}, "
    f"Gold: {actual_revenue}"
)

print("PASS: Silver and Gold revenue reconciled")
print(f"Revenue: {actual_revenue:.2f}")


In [0]:

from pyspark.sql import functions as F

catalog = "adb_adventureworks_project"

fact = spark.table(f"{catalog}.gold.fact_sales")
monthly = spark.table(f"{catalog}.gold.agg_monthly_sales")

# Preserve full precision until the final comparison
fact_revenue = (
    fact.agg(F.sum("EstimatedRevenue").alias("revenue"))
    .first()["revenue"]
)

monthly_revenue = (
    monthly.agg(F.sum("EstimatedRevenue").alias("revenue"))
    .first()["revenue"]
)

assert fact_revenue == monthly_revenue, (
    f"Revenue mismatch: "
    f"fact_sales={fact_revenue}, "
    f"agg_monthly_sales={monthly_revenue}"
)

print("PASS: Gold fact and monthly aggregate revenue reconciled")
print(f"Estimated revenue: {fact_revenue:.2f}")
